<a href="https://colab.research.google.com/github/DymaStar/DTA_2026/blob/main/ML/DS100626OksanaLinearRegression_DTA_commented_UA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LinearRegression_DTA — прокоментована версія українською

У цій версії коментарі додані прямо в кодові блоки. Файл має формат `.ipynb`, тому відкривається в Jupyter Notebook, JupyterLab, VS Code та Google Colab.


In [ ]:
# ===== Блок 1: коментарі українською =====
# Імпортуємо бібліотеки для роботи з даними, графіками та машинним навчанням.
# ===========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Щоб результати були однаковими щоразу (відтворюваність)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Бібліотеки готові ✅")

У реальній роботі завантажте CSV через `pd.read_csv("file.csv")`.

**Генеруємо дані**  
Створюємо таблицю з ознаками й «справжньою» ціною (з невеликим випадковим шумом, як у житті).

In [ ]:
# ===== Блок 2: коментарі українською =====
# Переглядаємо перші рядки таблиці.
# ===========================================

n = 1000  # скільки квартир

area      = np.random.normal(60, 20, n).clip(20, 140)        # площа, м²
rooms     = np.clip(np.round(area / 25 + np.random.normal(0, 0.6, n)), 1, 5)  # кімнати
floor     = np.random.randint(1, 25, n)                      # поверх
dist_km   = np.random.exponential(5, n).clip(0.3, 25)        # відстань до центру, км
age_years = np.random.randint(0, 60, n)                      # вік будинку, років

# "Справжня" логіка ціни (у тис. $) — у житті її НЕ знаємо, модель має її відкрити:
price = (
    40                       # базова
    + area * 1.8             # кожен м² додає
    + rooms * 5              # кожна кімната
    + floor * 0.4            # трохи за поверх
    - dist_km * 3.0          # далі від центру — дешевше
    - age_years * 0.5        # старіший будинок — дешевше
    + np.random.normal(0, 12, n)   # шум: усе інше, що ми не врахували
).clip(20, None)

df = pd.DataFrame({
    "area": area.round(1),
    "rooms": rooms.astype(int),
    "floor": floor,
    "dist_km": dist_km.round(1),
    "age_years": age_years,
    "price": price.round(1),
})

# показуємо перші рядки таблиці
df.head()

In [ ]:
# ===== Блок 3: коментарі українською =====
# Дивимось типи даних і наявність пропущених значень.
# ===========================================

# перевіряємо типи даних і пропущені значення
df.info()

In [ ]:
# ===== Блок 4: коментарі українською =====
# Отримуємо коротку статистику по числових колонках.
# Перевіряємо пропущені значення.
# ===========================================

print(f"Розмір таблиці: {df.shape[0]} рядків х {df.shape[1]} стовпців")

print("\nЧи є пропуски?")
print(df.isna().sum())
# дивимось описову статистику
df.describe().round(2)

In [ ]:
# ===== Блок 5: коментарі українською =====
# Будуємо графік для візуального аналізу.
# ===========================================

# налаштовуємо або будуємо графік
plt.figure(figsize=(7, 4))
# налаштовуємо або будуємо графік
plt.scatter(df['area'], df['price'], alpha=0.3, s=15)
# налаштовуємо або будуємо графік
plt.xlabel("Площа (м²)")
# налаштовуємо або будуємо графік
plt.ylabel("Ціна (тис. $)")
# налаштовуємо або будуємо графік
plt.title("Чим більша площа - тим вища ціна (з розкидом)")
# налаштовуємо або будуємо графік
plt.tight_layout()
# виводимо графік на екран
plt.show()

In [ ]:
# ===== Блок 6: коментарі українською =====
# Імпортуємо бібліотеки для роботи з даними, графіками та машинним навчанням.
# X — це незалежні змінні / ознаки, які модель використовує для навчання.
# y — це цільова змінна, яку модель буде прогнозувати.
# Ділимо дані на тренувальну та тестову вибірки.
# ===========================================

# розділяємо дані на тренувальні і тестові
from sklearn.model_selection import train_test_split

X = df[['area', 'rooms', 'floor', 'dist_km', 'age_years']]  # features - ознаки
y = df['price']  # target - ціль

# розділяємо дані на тренувальні і тестові
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Навчальна вибірка:", X_train.shape[0], "квартир")
print("Тестова вибірка:", X_test.shape[0], "квартир")

## Лінійна регресія

| X | y | **X_train** | X_test | **y_train** | y_test | y_pred |
| - | - | ------- | ------ | ------- | ------ | -- |
| 1 | 1 | 1 |  | 1 |  |  |
| 2 | 4 | 2 |  | 4 |  |  |
| 3 | 9 |  | 3 |  | 9 | 90 |
| 4 | 16 | 4 |  | 16 |  |  |
| 5 | 25 |  | 5 |  | 25 | 250 |
| 6 | 36 | 6 |  | 36 |  |  |
| 7 | 49 | 7 |  | 49 |  |  |
| 8 | 64 | 8 |  | 64 |  |  |
| 9 | 81 | 9 |  | 81 |  |  |
| 10 | 100 | 10 |  | 100 |  |  |
| 11 |  |  |  |  |  | 1210 |

In [ ]:
# ===== Блок 7: коментарі українською =====
# Імпортуємо бібліотеки для роботи з даними, графіками та машинним навчанням.
# Створюємо модель лінійної регресії.
# Навчаємо модель на тренувальних даних.
# Робимо прогноз за допомогою навченої моделі.
# ===========================================

# створюємо модель лінійної регресії
from sklearn.linear_model import LinearRegression

# створюємо модель лінійної регресії
model = LinearRegression()
# навчаємо модель
model.fit(X_train, y_train)
# прогнозуємо значення
y_pred = model.predict(X_test)

compare = pd.DataFrame({
    "real_price": y_test[:5].round(2),
    "predict_price": y_pred[:5].round(2)
})
compare["error"] = (compare['predict_price'] - compare['real_price']).round(2)

compare

MAE - наскільки в середньому ми помилилися - 7 -> 7 тис. $

RMSE (корінь з MSE) - схожу на MAE, сильніше карає великі промахи - менше = краще

R² - яку частину розкиду модель може оцінити - [0.;1.0] -> 1 - ідеально, 0 - вгадує середній показник

In [ ]:
# ===== Блок 8: коментарі українською =====
# Імпортуємо бібліотеки для роботи з даними, графіками та машинним навчанням.
# Оцінюємо якість моделі за допомогою метрик.
# ===========================================

# R² показує, наскільки добре модель пояснює дані
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MAE — середня абсолютна помилка
mae = mean_absolute_error(y_test, y_pred)

# mse = mean_squared_error(y_test, y_pred)
# MSE/RMSE — помилка з урахуванням квадратів відхилень
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R² показує, наскільки добре модель пояснює дані
r2 = r2_score(y_test, y_pred)

print(f"MAE = {mae:.1f} тис. $ (у середньому помиляємося на стільки)")
# print(f"MSE = {mse:.1f}")
print(f"RMSE = {rmse:.1f} тис. $")
print(f"R² = {r2:.3f} (модель пояснює {r2 * 100:.1f}% розкиду ціни)")

In [ ]:
# ===== Блок 9: коментарі українською =====
# y — це цільова змінна, яку модель буде прогнозувати.
# ===========================================

coefs = pd.DataFrame({
    "features": X.columns,
    "coef": model.coef_.round(2)
}).sort_values("coef", key=abs, ascending=False)

print(f"Базова ціна: {model.intercept_:.2F} тис. $\n")
print("Як кожна ознака вприлаває на ціну")
coefs

In [ ]:
# ===== Блок 10: коментарі українською =====
# Цей блок виконує один із кроків аналізу даних або побудови моделі.
# ===========================================

price2 = (
    40.83
    + area *	1.78
    + rooms * 5.22
    + dist_km	* (-3.07)
    + age_years	* (-0.49)
    + floor * (0.39)
)

In [ ]:
# ===== Блок 11: коментарі українською =====
# Цей блок виконує один із кроків аналізу даних або побудови моделі.
# ===========================================

# "Справжня" логіка ціни (у тис. $) — у житті її НЕ знаємо, модель має її відкрити:
price = (
    40                       # базова
    + area * 1.8             # кожен м² додає
    + rooms * 5              # кожна кімната
    + floor * 0.4            # трохи за поверх
    - dist_km * 3.0          # далі від центру — дешевше
    - age_years * 0.5        # старіший будинок — дешевше
    + np.random.normal(0, 12, n)   # шум: усе інше, що ми не врахували
).clip(20, None)